In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

In [2]:
#Settings
# ==========================================
IMAGE_DIR = r"D:\COURSE_DATA\Intro_Deep_Learning\project\data\Post_Impressionism"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# best known hyperparameters
LEARNING_RATE = 0.0002938
BATCH_SIZE = 32
OPTIMIZER_NAME = 'Adam'
NUM_EPOCHS = 5

print(f"✅ Device: {DEVICE}")
print(f"🚀 Starting Final Training with: LR={LEARNING_RATE}, Batch={BATCH_SIZE}, Epochs={NUM_EPOCHS}")

✅ Device: cuda
🚀 Starting Final Training with: LR=0.0002938, Batch=32, Epochs=5


In [3]:
# Data preparation
# ==========================================
def prepare_data(dir_path):
    if not os.path.exists(dir_path):
        print(f"❌ Error: Folder not found at {dir_path}")
        exit()

    all_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labels = [1 if f.lower().startswith("vincent-van-gogh") else 0 for f in all_files]

    print(f"📂 Total images found: {len(all_files)}")

    #  Train (80%), Test (10%), Validation (10%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        all_files, labels, test_size=0.10, random_state=42, stratify=labels
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(IMAGE_DIR)
print(f"📊 Split: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

📂 Total images found: 6450
📊 Split: Train=5160 | Val=645 | Test=645


In [5]:
#  Dataset & Transforms
# ==========================================
class SimpleFolderDataset(Dataset):
    def __init__(self, filenames, labels, root_dir, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.filenames[idx])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]



train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


train_loader = DataLoader(SimpleFolderDataset(X_train, y_train, IMAGE_DIR, train_transforms),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SimpleFolderDataset(X_val, y_val, IMAGE_DIR, val_transforms),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SimpleFolderDataset(X_test, y_test, IMAGE_DIR, val_transforms),
                         batch_size=BATCH_SIZE, shuffle=False)

In [6]:
# Model construction
# ==========================================
def get_model():
    print("🏗️  Building VGG19...")
    model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)

    #Features freezing
    for param in model.features.parameters():
        param.requires_grad = False

    # changing classifier
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)
    return model.to(DEVICE)


model = get_model()

# Setting loss weights
class_weights = torch.tensor([1.0, 5.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# optimizer setting
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

🏗️  Building VGG19...


In [7]:
# Training
# ==========================================
best_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")

    # Training
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc="Training")

    for imgs, lbls in loop:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc="Validating"):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    # Scores calculation
    val_f1 = f1_score(all_labels, all_preds, average='binary')
    val_acc = accuracy_score(all_labels, all_preds)
    try:
        val_auc = roc_auc_score(all_labels, all_probs)
    except:
        val_auc = 0.5
    cm = confusion_matrix(all_labels, all_preds)

    print(f"📊 Results: F1: {val_f1:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")
    print(f"🔲 Confusion Matrix: {cm.tolist()}")

    #Best known model (by F1 score
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_vangogh_vgg19_final.pth")
        print("💾 Model Saved (Best F1)")

print("\n✅ Training Complete!")


--- Epoch 1/5 ---


Validating: 100%|██████████| 21/21 [00:43<00:00,  2.07s/it]


📊 Results: F1: 0.5595 | Acc: 0.7705 | AUC: 0.9398
🔲 Confusion Matrix: [[403, 141], [7, 94]]
💾 Model Saved (Best F1)

--- Epoch 2/5 ---


Validating: 100%|██████████| 21/21 [00:46<00:00,  2.23s/it]


📊 Results: F1: 0.7500 | Acc: 0.9318 | AUC: 0.9428
🔲 Confusion Matrix: [[535, 9], [35, 66]]
💾 Model Saved (Best F1)

--- Epoch 3/5 ---


...

📊 Results: F1: 0.7327 | Acc: 0.9163 | AUC: 0.9576
🔲 Confusion Matrix: [[517, 27], [27, 74]]

--- Epoch 4/5 ---


Validating: 100%|██████████| 21/21 [01:15<00:00,  3.57s/it]


📊 Results: F1: 0.7300 | Acc: 0.9163 | AUC: 0.9494
🔲 Confusion Matrix: [[518, 26], [28, 73]]

--- Epoch 5/5 ---


Validating: 100%|██████████| 21/21 [01:04<00:00,  3.07s/it]

📊 Results: F1: 0.5981 | Acc: 0.8000 | AUC: 0.9547
🔲 Confusion Matrix: [[420, 124], [5, 96]]

✅ Training Complete!


In [8]:
#Testing
# ==========================================
print("\n🔍 Running Final Evaluation on Test Set...")
# Using best known weights configuration
model.load_state_dict(torch.load("best_vangogh_vgg19_final.pth"))
model.eval()

test_preds, test_labels, test_probs = [], [], []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Testing"):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, preds = torch.max(outputs, 1)

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(lbls.cpu().numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
test_acc = accuracy_score(test_labels, test_preds)
test_auc = roc_auc_score(test_labels, test_probs)
test_cm = confusion_matrix(test_labels, test_preds)

print("\n🏆 FINAL TEST RESULTS:")
print(f"   F1 Score: {test_f1:.4f}")
print(f"   Accuracy: {test_acc:.4f}")
print(f"   AUC-ROC:  {test_auc:.4f}")
print(f"   Confusion Matrix:\n{test_cm}")


🔍 Running Final Evaluation on Test Set...


Testing: 100%|██████████| 21/21 [03:04<00:00,  8.81s/it]


🏆 FINAL TEST RESULTS:
   F1 Score: 0.7391
   Accuracy: 0.9256
   AUC-ROC:  0.9392
   Confusion Matrix:
[[529  16]
 [ 32  68]]
